# Temporal Convolutional Networks (TCN) for Power Forecasting with TensorBoard

**Task:** Predict building electrical power consumption 24 hours ahead (144 steps at 10-minute resolution) using a PyTorch Temporal Convolutional Network (TCN) with causal dilated convolutions.  
**Data:** `library_featured_v2.csv` (engineered in notebook 05) with 27 temporal, calendar, occupancy, and autoregressive features.  
**Benchmark Target:** Compare parallel dilated convolutions against sequential BiLSTM, tabular MLP, and tree models (LightGBM, Extra Trees, XGBoost).

---

### Why TCN for University Building Management Systems (BMS)?
While recurrent networks (LSTM/GRU) process sequences sequentially, TCNs introduce key advantages for building telemetry:
1. **Massive Parallelism:** Since convolutions are computed across all timesteps simultaneously rather than sequentially, training and inference execute significantly faster on both CPU and GPU.
2. **Strict Causality (Zero Future Leakage):** A causal convolution guarantees that the forecast at time $t$ depends strictly on timesteps $\le t$. We achieve this via asymmetric left-padding: $\text{pad}_{\text{left}} = (K - 1) \cdot d$, followed by a `Chomp1d` layer removing trailing elements.
3. **Exponential Receptive Field via Dilations:** By stacking residual blocks with exponentially increasing dilation factors ($d = 1, 2, 4$), the model's receptive field expands exponentially without exploding parameter count:
   $$\text{Receptive Field} = 1 + 2 \cdot (K - 1) \sum_{i=0}^{N-1} d_i$$
4. **Residual Connections:** Each block incorporates an identity skip connection ($y = \text{Activation}(x + \mathcal{F}(x))$), preventing vanishing or exploding gradients over deep temporal backbones.
5. **Continuous Linear Output:** The final temporal representation is projected via `Linear(64)` $\to$ `ReLU` $\to$ `Dropout(0.2)` $\to$ `Linear(1)`. The output has **no activation function**, producing unconstrained physical power forecasts in Watts ($W$).

## 1. Imports & Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure utils path is discoverable
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
if "." not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from utils.tb_logger import TorchTBLogger

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Computation Device: {DEVICE}")

PyTorch Version: 2.14.0+cpu
Computation Device: cpu


## 2. Load Engineered Feature Matrix

In [2]:
DATA_PATH = "library_featured_v2.csv" if os.path.exists("library_featured_v2.csv") else "../library_featured_v2.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded library_featured_v2.csv: 41,248 rows x 37 columns


## 3. Feature Selection & Chronological Train/Val/Test Split

In [3]:
FEATURE_COLUMNS = [
    # Baseline power & occupancy
    "power", "occupancy_count", "occupancy_roll_mean_6", "occupancy_diff_1", "rooms_occupied_est",
    # Cyclical temporal features
    "hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos", "dayofyear_sin", "dayofyear_cos",
    # BMS operational schedule flags
    "weekend", "is_class_hour", "is_evening_study", "is_night", "is_exam_month", "is_vacation_month",
    # Autoregressive lags & rolling stats
    "power_lag_144", "power_lag_288", "power_lag_1008", "occupancy_lag_144",
    "power_roll_mean_144", "power_roll_std_144", "power_roll_min_144", "power_roll_max_144"
]
TARGET_COLUMN = "target"

# Primary 70 / 30 chronological split
test_split_idx = int(len(df) * 0.70)
train_val_df = df.iloc[:test_split_idx].reset_index(drop=True)
test_df = df.iloc[test_split_idx:].reset_index(drop=True)

# Train / Validation sub-split (85% train / 15% val)
val_split_idx = int(len(train_val_df) * 0.85)
train_df = train_val_df.iloc[:val_split_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_split_idx:].reset_index(drop=True)

print(f"Features: {len(FEATURE_COLUMNS)} variables")
print(f"Train sub-split: {len(train_df):>6,} rows | {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Validation split:{len(val_df):>6,} rows | {val_df['timestamp'].min()} to {val_df['timestamp'].max()}")
print(f"Test split:      {len(test_df):>6,} rows | {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")

Features: 27 variables
Train sub-split: 24,542 rows | 2014-02-26 10:30:00+05:30 to 2016-08-10 12:00:00+05:30
Validation split: 4,331 rows | 2016-08-10 12:10:00+05:30 to 2016-11-20 12:50:00+05:30
Test split:      12,375 rows | 2016-11-20 13:00:00+05:30 to 2017-11-02 23:50:00+05:30


## 4. Normalization and 1D Convolutional Sequence Formatting

TCN expects inputs of shape:
$$(B, C_{\text{in}}, L)$$
where $B$ is batch size, $C_{\text{in}} = 27$ is the number of feature channels, and $L = 36$ is the temporal lookback sequence length.

In [4]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_x.fit_transform(train_df[FEATURE_COLUMNS].values)
y_train_scaled = scaler_y.fit_transform(train_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_val_scaled = scaler_x.transform(val_df[FEATURE_COLUMNS].values)
y_val_scaled = scaler_y.transform(val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_x.transform(test_df[FEATURE_COLUMNS].values)
y_test_scaled = scaler_y.transform(test_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel()

SEQ_LEN = 36

class TCNSequenceDataset(Dataset):
    def __init__(self, X, y, seq_len=36):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.X) - self.seq_len + 1
        
    def __getitem__(self, idx):
        # Slice [seq_len, D] and transpose to [D, seq_len] for Conv1d
        x_window = self.X[idx : idx + self.seq_len].transpose(0, 1)
        y_target = self.y[idx + self.seq_len - 1]
        return x_window, y_target

BATCH_SIZE = 256

train_dataset = TCNSequenceDataset(X_train_scaled, y_train_scaled, seq_len=SEQ_LEN)
val_dataset   = TCNSequenceDataset(X_val_scaled,   y_val_scaled,   seq_len=SEQ_LEN)
test_dataset  = TCNSequenceDataset(X_test_scaled,  y_test_scaled,  seq_len=SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"TCN Datasets ready (seq_len={SEQ_LEN}):")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Val:   {len(val_dataset):,} samples")
print(f"  Test:  {len(test_dataset):,} samples")

TCN Datasets ready (seq_len=36):
  Train: 24,507 samples
  Val:   4,296 samples
  Test:  12,340 samples


## 5. PyTorch TCN Architecture Implementation

We build:
1. `Chomp1d`: Trims trailing padded elements to enforce strict temporal causality.
2. `TemporalBlock`: Dilated Conv1d $\to$ Chomp1d $\to$ BatchNorm $\to$ ReLU $\to$ Dropout $\to$ Dilated Conv1d $\to$ Chomp1d $\to$ BatchNorm $\to$ ReLU $\to$ Dropout, plus a $1 \times 1$ conv residual path when channel counts differ.
3. `TemporalConvNet`: Stacks temporal blocks with dilations ($d = 1, 2, 4$).
4. `BMS_TCN`: Full model taking $(B, C_{\text{in}}, L)$, passing through TCN, extracting the final timestep feature vector, and applying a linear regression projection.

In [5]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.bn1 = nn.BatchNorm1d(n_outputs)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.bn2 = nn.BatchNorm1d(n_outputs)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.bn1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.bn2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class BMS_TCN(nn.Module):
    def __init__(self, num_inputs, num_channels=[64, 64, 64], kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]
            padding = (kernel_size - 1) * dilation_size
            layers.append(TemporalBlock(
                in_channels, out_channels, kernel_size,
                stride=1, dilation=dilation_size, padding=padding, dropout=dropout
            ))
        self.tcn = nn.Sequential(*layers)
        
        # Regression head on the final timestep
        self.head = nn.Sequential(
            nn.Linear(num_channels[-1], 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Linear output (no activation)
        )

    def forward(self, x):
        # x: [B, C_in, L]
        y = self.tcn(x)
        # Extract representation from the final timestep
        last_step = y[:, :, -1]
        out = self.head(last_step)
        return out.squeeze(-1)

model = BMS_TCN(num_inputs=len(FEATURE_COLUMNS), num_channels=[64, 64, 64], kernel_size=3, dropout=0.2).to(DEVICE)
print(model)
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

BMS_TCN(
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(27, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp1): Chomp1d()
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu1): ReLU()
      (dropout1): Dropout(p=0.2, inplace=False)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp2): Chomp1d()
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu2): ReLU()
      (dropout2): Dropout(p=0.2, inplace=False)
      (net): Sequential(
        (0): Conv1d(27, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (1): Chomp1d()
        (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (3): ReLU()
        (4): Dropout(p=0.2, inplace=False)
        (5): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,))
        (6): Chomp1d()
        (7):

## 6. Initialize TensorBoard Logger & Computational Graph

In [6]:
log_root = "../runs" if os.path.basename(os.getcwd()) == "notebooks" else "runs"
os.makedirs(log_root, exist_ok=True)

sample_x, _ = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)

tb = TorchTBLogger(
    model_name="TCN",
    model=model,
    sample_input=sample_x,
    log_root=log_root
)

[TB] 'TCN' -> ../runs\TCN
[TB] Launch: tensorboard --logdir=../runs/


[TB] Model graph logged.


## 7. Model Training & Per-Epoch TensorBoard Logging

In [7]:
EPOCHS = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()

best_val_loss = float("inf")
checkpoint_path = os.path.join(log_root, "best_TCN.pt")

print(f"Training TCN on {DEVICE} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        
    avg_train_loss = np.mean(train_losses)
    
    model.eval()
    val_losses = []
    val_preds_list = []
    val_targets_list = []
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            val_losses.append(loss.item())
            val_preds_list.extend(preds.cpu().numpy())
            val_targets_list.extend(y_b.cpu().numpy())
            
    avg_val_loss = np.mean(val_losses)
    current_lr = scheduler.get_last_lr()[0]
    
    val_preds_W = scaler_y.inverse_transform(np.array(val_preds_list).reshape(-1, 1)).ravel()
    val_targets_W = scaler_y.inverse_transform(np.array(val_targets_list).reshape(-1, 1)).ravel()
    
    val_mae = mean_absolute_error(val_targets_W, val_preds_W)
    val_rmse = np.sqrt(mean_squared_error(val_targets_W, val_preds_W))
    val_r2 = r2_score(val_targets_W, val_preds_W)
    
    tb.log_epoch(
        epoch=epoch,
        train_loss=avg_train_loss,
        val_loss=avg_val_loss,
        y_val=val_targets_W,
        y_pred=val_preds_W,
        lr=current_lr
    )
    
    if epoch % 5 == 0:
        tb.log_gradients(model, epoch)
        
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        
    scheduler.step()
    
    print(f"Epoch {epoch:>2d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val MAE: {val_mae:,.1f} W | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")

print(f"\nTraining complete. Best checkpoint saved to {checkpoint_path}")

Training TCN on cpu for 15 epochs...


Epoch  1/15 | Train Loss: 0.3841 | Val Loss: 0.4020 | Val MAE: 2,948.7 W | Val R²: 0.6505 | LR: 1.00e-03


Epoch  2/15 | Train Loss: 0.2197 | Val Loss: 0.3846 | Val MAE: 2,909.1 W | Val R²: 0.6652 | LR: 9.89e-04


Epoch  3/15 | Train Loss: 0.1720 | Val Loss: 0.4135 | Val MAE: 3,065.9 W | Val R²: 0.6398 | LR: 9.57e-04


Epoch  4/15 | Train Loss: 0.1532 | Val Loss: 0.3893 | Val MAE: 3,052.8 W | Val R²: 0.6610 | LR: 9.05e-04


...

Epoch 11/15 | Train Loss: 0.0957 | Val Loss: 0.4056 | Val MAE: 3,061.3 W | Val R²: 0.6465 | LR: 2.58e-04


Epoch 12/15 | Train Loss: 0.0927 | Val Loss: 0.4031 | Val MAE: 3,046.6 W | Val R²: 0.6487 | LR: 1.74e-04


Epoch 13/15 | Train Loss: 0.0896 | Val Loss: 0.3938 | Val MAE: 3,028.7 W | Val R²: 0.6568 | LR: 1.05e-04


Epoch 14/15 | Train Loss: 0.0885 | Val Loss: 0.3950 | Val MAE: 3,025.4 W | Val R²: 0.6558 | LR: 5.28e-05


Epoch 15/15 | Train Loss: 0.0874 | Val Loss: 0.3971 | Val MAE: 3,047.9 W | Val R²: 0.6540 | LR: 2.08e-05

Training complete. Best checkpoint saved to ../runs\best_TCN.pt


## 8. Final Test Evaluation & Benchmark Comparison

In [8]:
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

# Predict on test set
test_preds_list = []
with torch.no_grad():
    for x_b, _ in test_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        test_preds_list.extend(preds.cpu().numpy())

test_preds_W = scaler_y.inverse_transform(np.array(test_preds_list).reshape(-1, 1)).ravel()
test_targets_W = test_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

# Predict on train set for gap check
train_val_dataset = TCNSequenceDataset(
    scaler_x.transform(train_val_df[FEATURE_COLUMNS].values),
    scaler_y.transform(train_val_df[TARGET_COLUMN].values.reshape(-1, 1)).ravel(),
    seq_len=SEQ_LEN
)
train_val_loader = DataLoader(train_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_preds_list = []
with torch.no_grad():
    for x_b, _ in train_val_loader:
        x_b = x_b.to(DEVICE)
        preds = model(x_b)
        train_preds_list.extend(preds.cpu().numpy())

train_preds_W = scaler_y.inverse_transform(np.array(train_preds_list).reshape(-1, 1)).ravel()
train_targets_W = train_val_df[TARGET_COLUMN].values[SEQ_LEN - 1:]

test_mae = mean_absolute_error(test_targets_W, test_preds_W)
test_rmse = np.sqrt(mean_squared_error(test_targets_W, test_preds_W))
test_r2 = r2_score(test_targets_W, test_preds_W)
train_r2 = r2_score(train_targets_W, train_preds_W)
r2_gap = train_r2 - test_r2

print("=" * 55)
print(f"  PyTorch TCN Final Benchmark Results (Test Set)")
print("=" * 55)
print(f"  Test MAE:   {test_mae:,.1f} W")
print(f"  Test RMSE:  {test_rmse:,.1f} W")
print(f"  Test R²:    {test_r2:.4f}")
print(f"  Train R²:   {train_r2:.4f}")
print(f"  R² Gap:     {r2_gap:.4f}")
print("=" * 55)

  PyTorch TCN Final Benchmark Results (Test Set)
  Test MAE:   4,058.0 W
  Test RMSE:  5,557.1 W
  Test R²:    0.5124
  Train R²:   0.8286
  R² Gap:     0.3162


## 9. Log Final Metrics, Prediction Plots & HParams to TensorBoard

In [9]:
tb.log_final(
    y_train=train_targets_W,
    y_pred_train=train_preds_W,
    y_test=test_targets_W,
    y_pred_test=test_preds_W,
    hparams={
        "architecture": "TCN (3 levels, 64-ch, dilations=[1,2,4])",
        "seq_len": SEQ_LEN,
        "kernel_size": 3,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_init": 1e-3,
        "dropout": 0.2,
        "optimizer": "AdamW",
        "output_act": "Linear"
    }
)

tb.log_prediction_plots(y_test=test_targets_W, y_pred=test_preds_W, n_plot=500)
tb.log_scatter_plot(y_test=test_targets_W, y_pred=test_preds_W)

tb.close()
print("TensorBoard logging complete. Logs written to:", log_root)


[TCN] Train  MAE=1982W  R2=0.829
[TCN] Test   MAE=4058W  R2=0.512


[TB] Writer closed: 'TCN'
TensorBoard logging complete. Logs written to: ../runs


## 10. Committee Presentation Summary: Model Comparison

| Model | Model Family | Test MAE (W) | Test RMSE (W) | Test R² | Train-Test R² Gap |
|---|---|---|---|---|---|
| **LightGBM** | Gradient Boosted Trees | ~3,340 | ~4,580 | ~0.672 | 0.210 |
| **Extra Trees** | Randomized Ensembles | ~3,410 | ~4,690 | ~0.655 | 0.280 |
| **XGBoost** | Gradient Boosted Trees | 3,180 | 4,405 | 0.693 | 0.170 |
| **PyTorch MLP** | Deep Tabular NN | ~3,469 | ~4,636 | ~0.660 | 0.168 |
| **PyTorch BiLSTM** | Sequential Recurrent NN | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* |
| **PyTorch TCN** | Dilated Causal Convolutional | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* | *(Logged in TB)* |

Launch TensorBoard to inspect side-by-side:
```bash
tensorboard --logdir=runs/
```